# 8. Мини-проект. Практика

## Постановка задачи

Мы выступим в роли программиста-сомелье: нам предстоит определять качество вина. Необходимо решить задачу классификации с использованием SVM и подобрать наилучшее ядро.

## Данные

Датасет содержит информацию о красных винах и их составе. Целевой переменной является столбец "quality". Это метрика качества вина по шкале от 3 до 8.

## Рекомендации

1. Проанализируйте распределения переменных и корреляцию с таргетом.
2. Сделайте целевую переменную категориальной ('bad wine': quality < 6.5 и 'good wine': quality > 6.5).
3. Закодируйте целевую переменную.
4. Используйте StandardScaler() для преобразования признаков.
5. Настройте гиперпараметры модели SVC: C, gamma и kernel из ['linear', 'poly', 'rbf', 'sigmoid'].
6. Используйте метрику accuracy. Также можно смотреть результаты confusion matrix.
7. Обучите модель с наилучшими параметрами и оцените score на кросс-валидации. 

## Критерии оценивания

- 2 балла	Сделан анализ (проанализированы распределения).
- 2 балла	Добавлены какие-либо новые признаки/ категоризирован таргет.
- 4 балла	С помощью одного из методов подбора гиперпараметров выбрано ядро.
- 2 балла	Обучена модель и получено значение метрики на валидации.

# Homework

## Импорт библиотек
Сначала импортируем нужные нам пакеты. Затем мы импортируем данные. Поскольку мы хотим знать взаимосвязь между логарифмической ошибкой и плотностью участков, для анализа нам нужны только строки из обучающего набора, содержащие координаты x, y. К остальным данным вернемся позже.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

from skopt import BayesSearchCV
from skopt.space import Real, Categorical

## 1. Подгрузим данные:

In [2]:
df = pd.read_csv('data_add4/winequality-red.csv', sep=';')
df.head(3)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5


Создадим категориальный признак 'good wine', условие 'quality' >= 6.5 вино хорошее, тогда good_wine = 1

In [3]:
# Сделаем целевую переменную категориальной ('bad wine': quality < 6.5 и 'good wine': quality > 6.5)
df['good_wine'] = (df['quality'] >= 6.5).astype(int)
df = df.drop(columns=['quality'])
df.head(3)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,good_wine
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,0


## 2. Анализ распределения переменных


In [4]:
# Базовая проверка структуры и пропусков
target_col = 'good_wine'
num_cols = df.select_dtypes(include='number').columns.tolist()
feature_cols = [col for col in num_cols if col != target_col]

print(f"Размер df: {df.shape}")
print("\nПропуски по столбцам:")
print(df[feature_cols + [target_col]].isna().sum().sort_values(ascending=False))

# Описательные статистики по числовым признакам
summary_stats = (
    df[feature_cols]
    .describe()
    .T
    .sort_values('std', ascending=False)
)
summary_stats

# Распределение целевой переменной
class_counts = (
    df[target_col]
    .value_counts()
    .sort_index()
    .rename(index={0: 'bad_wine', 1: 'good_wine'})
    .reset_index()
)
class_counts.columns = ['class', 'count']

fig_target = px.bar(
    class_counts,
    x='class',
    y='count',
    title='Распределение целевой переменной good_wine',
    text='count',
    color='class',
    color_discrete_sequence=['#d95f02', '#1b9e77']
)
fig_target.update_layout(showlegend=False)
fig_target.show()

# Гистограммы распределений всех числовых признаков
long_df = df[feature_cols].melt(var_name='feature', value_name='value')
fig_hist = px.histogram(
    long_df,
    x='value',
    facet_col='feature',
    facet_col_wrap=3,
    nbins=30,
    opacity=0.85,
    title='Распределения числовых признаков'
)
fig_hist.update_xaxes(matches=None)
fig_hist.update_yaxes(matches=None)
fig_hist.update_layout(height=1200)
fig_hist.show()

# Boxplot по классам для наиболее вариативных признаков
top6_var_features = summary_stats.head(6).index.tolist()
long_top = df[[target_col] + top6_var_features].melt(
    id_vars=target_col,
    var_name='feature',
    value_name='value'
)
long_top[target_col] = long_top[target_col].map({0: 'bad_wine', 1: 'good_wine'})

fig_box = px.box(
    long_top,
    x='feature',
    y='value',
    color=target_col,
    points=False,
    title='Сравнение распределений по классам (топ-6 признаков по std)',
    color_discrete_map={'bad_wine': '#d95f02', 'good_wine': '#1b9e77'}
)
fig_box.show()

Размер df: (1599, 12)

Пропуски по столбцам:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
good_wine               0
dtype: int64


## 3. Feature Engineering

Добавлен новый признак:
- Отношение свободного к общему SO₂: **so2_ratio = free sulfur dioxide / (total sulfur dioxide + 1e-6)**

Была идея использовать и другие , но они создавали сильную Мультиколлинеарность и были удалены из дата сета в дальнейшем. Показываю для общего осведомления:
- Суммарная кислотность: **total_acidity = fixed acidity + volatile acidity + citric acid**
- Алкоголь к плотности: **alcohol_density_ratio = alcohol / (density + 1e-6)**
- Баланс кислотности и pH: **acidity_ph_interaction = fixed acidity * pH**

In [5]:
# 1) Отношение свободного к общему SO2
df['so2_ratio'] = df['free sulfur dioxide'] / (df['total sulfur dioxide'] + 1e-6)

# Проверка: вывод новых признаков
new_features = ['so2_ratio']
df[new_features].head(3)

,so2_ratio
0,0.323529
1,0.373134
2,0.277778


## 4. Корреляция с таргетом (quality)

In [6]:
# Correlation heatmap (Plotly)
corr_matrix = df.corr(numeric_only=True)

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Correlation Heatmap for Data"
)
fig.update_layout(width=1100, height=900)
fig.show()

# Top feature correlations with good_wine (Plotly)
activity_corr = corr_matrix["good_wine"].sort_values(ascending=False)
top_corr = activity_corr.drop(labels=["good_wine"], errors="ignore").head(20)
top_corr_df = top_corr.to_frame().T

top_fig = px.imshow(
    top_corr_df,
    text_auto=".3f",
    color_continuous_scale="RdYlBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Top Feature Correlations with good_wine"
)
top_fig.update_xaxes(side="bottom", tickangle=45)
top_fig.update_layout(height=380, margin=dict(l=40, r=40, t=80, b=140))
top_fig.show()

> Вывод по EDA:
> - Датасет полный: пропусков по признакам и таргету нет.
> - Целевая переменная несбалансирована: плохих вин заметно больше, чем хороших (примерно 86% против 14%).
> - По гистограммам видно, что часть признаков имеет асимметрию и выбросы, поэтому масштабирование перед SVM действительно необходимо.
> 
> Корреляция с таргетом
> - Наиболее положительная связь с good_wine у alcohol, затем у sulphates и citric acid.
> - Наиболее отрицательная связь у volatile acidity, далее у density, chlorides и total sulfur dioxide.
> - Сильных линейных связей в целом немного, поэтому для классификации имеет смысл подбирать нелинейные ядра SVM

## 5. Используем StandardScaler() для преобразования признаков

In [7]:
from sklearn.preprocessing import StandardScaler

# Отделяем признаки от целевой переменной
target_col = 'good_wine'
X = df.drop(columns=[target_col])
y = df[target_col]

# Масштабируем только признаки
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Собираем обратно DataFrame: признаки scaled + исходный таргет
df_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=df.index)
df_scaled[target_col] = y.values

df_scaled.head(3)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,so2_ratio,good_wine
0,-0.528360,0.961877,-1.391472,-0.453218,-0.243707,-0.466193,-0.379133,0.558274,1.288643,-0.579207,-0.960246,-0.379847,0
1,-0.298547,1.967442,-1.391472,0.043416,0.223875,0.872638,0.624363,0.028261,-0.719933,0.128950,-0.584777,-0.059300,0
2,-0.298547,1.297065,-1.186070,-0.169427,0.096353,-0.083669,0.229047,0.134264,-0.331177,-0.048089,-0.584777,-0.675494,0


# 6. Выбор ядра и настройка гиперпараметров. Обучение моделей и валидация

- Выберем ядро методом байесовской оптимизации (BayesSearchCV)
- Обучим модель, получим метрики на валидации и кросс-валидации

In [8]:
# Используем масштабированные признаки из шага 5
target_col = 'good_wine'
X_model = df_scaled.drop(columns=[target_col])
y_model = df_scaled[target_col]

# Hold-out для оценки confusion matrix
X_train, X_valid, y_train, y_valid = train_test_split(
    X_model, y_model, test_size=0.2, random_state=42, stratify=y_model
)

# Байесовская оптимизация гиперпараметров SVC
search_spaces = {
    'C': Real(1e-2, 1e2, prior='log-uniform'),
    'gamma': Real(1e-3, 1e1, prior='log-uniform'),
    'kernel': Categorical(['linear', 'poly', 'rbf', 'sigmoid'])
}

bayes_search = BayesSearchCV(
    estimator=SVC(),
    search_spaces=search_spaces,
    n_iter=40,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    random_state=42
)

bayes_search.fit(X_train, y_train)

best_svc = bayes_search.best_estimator_
y_pred = best_svc.predict(X_valid)

# Метрика accuracy на валидации
valid_accuracy = accuracy_score(y_valid, y_pred)
print('Best params:', bayes_search.best_params_)
print(f'Best CV accuracy (BayesSearchCV): {bayes_search.best_score_:.4f}')
print(f'Validation accuracy: {valid_accuracy:.4f}')

# Confusion matrix
cm = confusion_matrix(y_valid, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['actual_bad(0)', 'actual_good(1)'],
    columns=['pred_bad(0)', 'pred_good(1)']
)
print('\nConfusion matrix:')
display(cm_df)

# Оценка лучшей модели на кросс-валидации по всей выборке
cv_scores = cross_val_score(best_svc, X_model, y_model, cv=5, scoring='accuracy')
print('Cross-val accuracy scores:', np.round(cv_scores, 4))
print(f'Cross-val mean accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')


# Набор моделей для сравнения (включая лучшую SVC из шага 6)
models = {
    'LogisticRegression': LogisticRegression(max_iter=3000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42),
    'SVC_best': best_svc
}

metrics_rows = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)

    acc = accuracy_score(y_valid, pred)
    prec = precision_score(y_valid, pred, zero_division=0)
    rec = recall_score(y_valid, pred, zero_division=0)
    f1 = f1_score(y_valid, pred, zero_division=0)

    cv_acc = cross_val_score(model, X_model, y_model, cv=5, scoring='accuracy')

    metrics_rows.append({
        'model': model_name,
        'val_accuracy': acc,
        'val_precision': prec,
        'val_recall': rec,
        'val_f1': f1,
        'cv_accuracy_mean': cv_acc.mean(),
        'cv_accuracy_std': cv_acc.std()
    })

comparison_df = pd.DataFrame(metrics_rows).sort_values('cv_accuracy_mean', ascending=False)
comparison_df = comparison_df.reset_index(drop=True)

print()
print('Сравнение метрик моделей:')
display(comparison_df.round(4))

# Визуализация 1: accuracy на валидации и CV
plot_df = comparison_df.melt(
    id_vars='model',
    value_vars=['val_accuracy', 'cv_accuracy_mean'],
    var_name='metric',
    value_name='score'
)

fig_cmp_acc = px.bar(
    plot_df,
    x='model',
    y='score',
    color='metric',
    barmode='group',
    text='score',
    title='Сравнение accuracy по моделям'
)
fig_cmp_acc.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_cmp_acc.update_layout(yaxis_range=[0.7, 1.0])
fig_cmp_acc.show()

# Визуализация 2: precision/recall/F1 на валидации
plot_df_prf = comparison_df.melt(
    id_vars='model',
    value_vars=['val_precision', 'val_recall', 'val_f1'],
    var_name='metric',
    value_name='score'
)

fig_cmp_prf = px.bar(
    plot_df_prf,
    x='model',
    y='score',
    color='metric',
    barmode='group',
    text='score',
    title='Сравнение precision / recall / F1 (validation)'
)
fig_cmp_prf.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_cmp_prf.update_layout(yaxis_range=[0.0, 1.0])
fig_cmp_prf.show()

Best params: OrderedDict([('C', 99.93242189807779), ('gamma', 0.6084342505025045), ('kernel', 'rbf')])
Best CV accuracy (BayesSearchCV): 0.9062
Validation accuracy: 0.9219

Confusion matrix:


,pred_bad(0),pred_good(1)
actual_bad(0),272,5
actual_good(1),20,23


Cross-val accuracy scores: [0.8688 0.8562 0.8562 0.8188 0.8683]
Cross-val mean accuracy: 0.8537 +/- 0.0183

Сравнение метрик моделей:


,model,val_accuracy,val_precision,val_recall,val_f1,cv_accuracy_mean,cv_accuracy_std
0,RandomForest,0.9406,0.9286,0.6047,0.7324,0.8749,0.0156
1,LogisticRegression,0.8906,0.6818,0.3488,0.4615,0.8656,0.0231
2,KNN,0.9000,0.6774,0.4884,0.5676,0.8562,0.0355
3,SVC_best,0.9219,0.8214,0.5349,0.6479,0.8537,0.0183


> ## Результаты и выводы:
>
> - Подбор гиперпараметров SVC выполнен через **BayesSearchCV** (байесовская оптимизация).
> - Лучшие параметры SVC: **C = 99.9324**, **gamma = 0.6084**, **kernel = rbf**.
> - Лучшее качество в оптимизации: **Best CV accuracy = 0.9062**.
> - Качество на валидации: **Validation accuracy = 0.9219**.
>
> - Confusion Matrix:
>   - actual_bad(0): pred_bad = **272**, pred_good = **5**
>   - actual_good(1): pred_bad = **20**, pred_good = **23**
>
> - Кросс-валидация лучшей SVC (5-fold): **[0.8688, 0.8562, 0.8562, 0.8188, 0.8683]**
> - Средняя CV accuracy лучшей SVC: **0.8537 +/- 0.0183**.
>
> - Сравнение моделей (validation / CV):
>   - **RandomForest**: val_accuracy = **0.9406**, precision = **0.9286**, recall = **0.6047**, f1 = **0.7324**, cv_accuracy_mean = **0.8749**
>   - **LogisticRegression**: val_accuracy = **0.8906**, precision = **0.6818**, recall = **0.3488**, f1 = **0.4615**, cv_accuracy_mean = **0.8656**
>   - **KNN**: val_accuracy = **0.9000**, precision = **0.6774**, recall = **0.4884**, f1 = **0.5676**, cv_accuracy_mean = **0.8562**
>   - **SVC (best)**: val_accuracy = **0.9219**, precision = **0.8214**, recall = **0.5349**, f1 = **0.6479**, cv_accuracy_mean = **0.8537**
>
> - Итог: по **cv_accuracy_mean** лучшей оказалась модель **RandomForest**, а оптимизированная **SVC** показала конкурентный результат и хорошее качество на валидации.